In [1]:
"""
Pooled A1 topomap LoRA training followed by patient-wise testing.

Workflow:
1. Load pooled A1 training metadata.
2. Create approximately 80% training / 20% validation per class.
3. Fine-tune MedGemma-4B with LoRA.
4. Select the best checkpoint using validation balanced accuracy.
5. Break BA ties using lower validation loss.
6. Early stop after 7 epochs without a strict BA improvement.
7. Reload the best adapter.
8. Evaluate Patients 08, 11, 18, and 24 separately.

Labels: 0 = non-seizure, 1 = seizure.
"""

from pathlib import Path
from datetime import datetime

import gc
import hashlib
import json
import math
import random
import re
import sys
import time

import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import AutoProcessor, get_linear_schedule_with_warmup

import peft
from peft import LoraConfig, get_peft_model, PeftModel

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    f1_score,
)

from tqdm import tqdm


# =====================================================================
# 1. PATHS
# =====================================================================

ROOT = Path(
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning"
)

TRAIN_CSV = (
    ROOT
    / "A1"
    / "Training"
    / "training_metadata.csv"
)

TEST_CSVS = {
    "08": (
        ROOT
        / r"Patient 08\A1\Testing\Blinded"
        / "Patient_08_A1_blinded_test_labels.csv"
    ),

    "11": (
        ROOT
        / r"Patient 11\A1\TRAINING_TESTING_SPLIT\Testing\Blinded Topomaps"
        / "patient11_A1_blinded_test.csv"
    ),

    "18": (
        ROOT
        / r"Patient 18\A1\Testing\Blinded folder"
        / "Patient_18_A1_blinded_test_labels.csv"
    ),

    "24": (
        ROOT
        / r"Patiient 24\A1\Testing\Blinded"
        / "Patient_24_A1_blinded_test_labels.csv"
    ),
}

OUTPUT_BASE = (
    ROOT
    / "A1"
    / "MedGemma4B_LoRA_Pooled_A1"
)


# =====================================================================
# 2. MODEL AND TRAINING SETTINGS
# =====================================================================

MODEL_ID = "google/medgemma-4b-it"

SEED = 42

MAX_EPOCHS = 50
PATIENCE = 7

VAL_FRACTION = 0.20

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

LR = 1e-4
WEIGHT_DECAY = 0.01

BATCH_SIZE = 1
GRAD_ACCUM = 8

MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05

MAX_NEW_TOKENS = 8
NUM_WORKERS = 0

# Optional exact column names if automatic detection fails.
#
# Example:
# COLUMN_OVERRIDES = {
#     "pooled": {"path": "image_path", "label": "label"},
#     "11": {"path": "image_path", "label": "label"},
# }

COLUMN_OVERRIDES = {}


# =====================================================================
# 3. PROMPTS
# =====================================================================

SYSTEM_PROMPT = """You are a strict vision-language classifier for EEG seizure detection.
Classify each EEG topomap image into exactly one of two labels:
Seizure
Non-seizure
Return only the final label. Do not provide explanation, punctuation, confidence score or extra text."""

USER_PROMPT = """You are an expert in identifying seizure patterns from EEG A1-coefficient variance topomaps. The attached image represents a 2-second EEG segment.
Classify the provided fixed-scale EEG A1-coefficient variance topomap as one of the following:
seizure
non-seizure
The raw EEG signal was divided into 2-second windows, and each window was filtered using a 60 Hz notch filter. A Discrete Wavelet Transform (DWT) was then applied separately to every bipolar EEG channel within each window, and the A1 approximation coefficients were extracted.
For each window, the variance of the A1 coefficients was calculated separately for every bipolar EEG channel. Each channel's A1-coefficient variance value was positioned at the midpoint between the two electrodes forming that bipolar channel. These values were spatially interpolated to create the EEG topomap. All topomaps use the same fixed color scale, allowing direct comparison across images. Yellow regions indicate higher A1-coefficient variance, while dark purple regions indicate lower A1-coefficient variance.
Classify the image based on the visible A1-coefficient variance intensity and its spatial distribution. Consider whether high-variance activity is focal or widespread and whether the pattern is symmetric or asymmetric. Evaluate these characteristics together rather than using any single feature as a definitive rule."""


# =====================================================================
# 4. GENERAL HELPERS
# =====================================================================

def save_json(path, data):
    path.write_text(
        json.dumps(
            data,
            indent=2,
            default=str,
            allow_nan=False,
        ),
        encoding="utf-8",
    )


def seed_all():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


def normalize_key(value):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).strip().lower(),
    )


def normalize_label(value):
    text = re.sub(
        r"[\s_\-–]+",
        "",
        str(value).strip().lower(),
    )

    if text in {
        "0",
        "0.0",
        "nonseizure",
        "normal",
        "interictal",
        "ns",
        "false",
    }:
        return 0

    if text in {
        "1",
        "1.0",
        "seizure",
        "ictal",
        "sz",
        "true",
    }:
        return 1

    raise ValueError(
        f"Unknown ground-truth label: {value!r}. "
        "Expected 0/1 or seizure/non-seizure."
    )


def pick_column(df, names, override=None, required=True):
    if override is not None:
        if override not in df.columns:
            raise ValueError(
                f"Column {override!r} missing. "
                f"Available columns: {list(df.columns)}"
            )
        return override

    lookup = {
        normalize_key(column): column
        for column in df.columns
    }

    for name in names:
        key = normalize_key(name)
        if key in lookup:
            return lookup[key]

    if required:
        raise ValueError(
            "Cannot identify metadata column. "
            f"Candidates: {names}\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


def safe_divide(numerator, denominator):
    if denominator == 0:
        return float("nan")
    return float(numerator / denominator)


# =====================================================================
# 5. COMPANION LABEL MERGE FOR BLIND CSVS
# =====================================================================

def find_label_column(df):
    return pick_column(
        df,
        [
            "standard_label",
            "label",
            "label_numeric",
            "true_label",
            "true_label_numeric",
            "ground_truth",
            "target",
            "class_label",
            "fine_tuning_class",
            "class_name",
            "class",
            "category",
            "selected_label",
            "selected_label_numeric",
        ],
        required=False,
    )


def companion_label_files(csv_path):
    folder = csv_path.parent
    stem = csv_path.stem

    candidates = []

    replacements = [
        ("_blinded_test", "_blinded_labels"),
        ("_blind_test", "_blind_labels"),
        ("_test", "_labels"),
        ("_test", "_metadata"),
    ]

    for old, new in replacements:
        if old in stem:
            candidates.append(folder / f"{stem.replace(old, new)}.csv")

    candidates.extend(sorted(folder.glob("*labels*.csv")))
    candidates.extend(sorted(folder.glob("*metadata*.csv")))

    unique = []
    seen = set()

    for candidate in candidates:
        if candidate == csv_path:
            continue

        key = str(candidate).lower()
        if key not in seen and candidate.exists():
            seen.add(key)
            unique.append(candidate)

    return unique


def attach_labels_from_companion(df, csv_path, dataset_name):
    if find_label_column(df) is not None:
        return df

    candidates = companion_label_files(csv_path)

    if not candidates:
        raise ValueError(
            f"{dataset_name}: no label column found in:\n{csv_path}\n\n"
            "For a blinded test CSV, keep the blind CSV label-free but place "
            "the matching labels/metadata CSV in the same folder. For Patient "
            "11 this is usually patient11_A1_blinded_labels.csv or "
            "patient11_A1_blinded_metadata.csv."
        )

    join_keys = [
        "blind_id",
        "image_filename",
        "blinded_image_filename",
        "filename",
        "global_serial",
        "global_serial_number",
        "Global_Serial",
    ]

    for label_file in candidates:
        label_df = pd.read_csv(label_file)
        label_df.columns = [str(column).strip() for column in label_df.columns]

        label_column = find_label_column(label_df)

        if label_column is None:
            continue

        for key in join_keys:
            left_key = pick_column(df, [key], required=False)
            right_key = pick_column(label_df, [key], required=False)

            if left_key is None or right_key is None:
                continue

            merged = df.merge(
                label_df[[right_key, label_column]].drop_duplicates(),
                left_on=left_key,
                right_on=right_key,
                how="left",
                suffixes=("", "_label_file"),
            )

            if merged[label_column].notna().all():
                print(
                    f"{dataset_name}: attached labels from "
                    f"{label_file.name} using key {key!r}."
                )
                return merged

    raise ValueError(
        f"{dataset_name}: label companion files were found, but labels "
        "could not be matched to the blinded test CSV.\n"
        f"Tried: {[str(path) for path in candidates]}"
    )


# =====================================================================
# 6. LOAD AND VALIDATE METADATA
# =====================================================================

def resolve_image_path(csv_path, value):
    if pd.isna(value) or not str(value).strip():
        raise ValueError(f"Missing image path in {csv_path}")

    image_path = Path(
        str(value).strip().strip('"').strip("'")
    )

    if not image_path.is_absolute():
        image_path = csv_path.parent / image_path

    if image_path.is_file():
        return image_path

    raise FileNotFoundError(
        f"Image not found:\n{image_path}\n"
        f"Source CSV:\n{csv_path}"
    )


def load_metadata(csv_path, dataset_name):
    csv_path = Path(csv_path)

    if not csv_path.is_file():
        raise FileNotFoundError(
            f"{dataset_name}: CSV not found:\n{csv_path}"
        )

    df = pd.read_csv(csv_path)

    if df.empty:
        raise ValueError(f"Empty CSV: {csv_path}")

    df.columns = [str(column).strip() for column in df.columns]

    override = COLUMN_OVERRIDES.get(dataset_name, {})

    df = attach_labels_from_companion(
        df,
        csv_path,
        dataset_name,
    )

    path_candidates = [
        "image_path",
        "fine_tuning_image_path",
        "selected_image_path",
        "file_path",
        "filepath",
        "path",
        "topomap_path",
        "filename",
        "image_filename",
    ]

    if dataset_name != "pooled":
        path_candidates = [
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "blinded_filename",
        ] + path_candidates

    image_column = pick_column(
        df,
        path_candidates,
        override.get("path"),
    )

    label_column = pick_column(
        df,
        [
            "standard_label",
            "label",
            "label_numeric",
            "true_label",
            "true_label_numeric",
            "ground_truth",
            "target",
            "class_label",
            "fine_tuning_class",
            "class_name",
            "class",
            "category",
            "selected_label",
            "selected_label_numeric",
        ],
        override.get("label"),
    )

    labels = df[label_column].map(normalize_label).astype(int)

    resolved_paths = []
    pixel_hashes = []

    for value in tqdm(
        df[image_column],
        desc=f"Validate images: {dataset_name}",
    ):
        image_path = resolve_image_path(csv_path, value)

        with Image.open(image_path) as source:
            rgb = source.convert("RGB")
            pixel_hash = hashlib.sha256(
                str(rgb.size).encode() + rgb.tobytes()
            ).hexdigest()

        resolved_paths.append(str(image_path.resolve()))
        pixel_hashes.append(pixel_hash)

    df = df.copy()

    df["image_path"] = resolved_paths
    df["label"] = labels
    df["class_name"] = df["label"].map(
        {0: "non_seizure", 1: "seizure"}
    )
    df["pixel_sha256"] = pixel_hashes
    df["metadata_source"] = str(csv_path)

    if set(df["label"]) != {0, 1}:
        raise ValueError(
            f"{dataset_name} must contain both classes "
            "for balanced accuracy."
        )

    print(
        f"\n{dataset_name}: {len(df)} images"
        f"\nClass counts: {df['label'].value_counts().to_dict()}"
    )

    return df


# =====================================================================
# 7. CHECK IDENTICAL IMAGES WITHIN AND ACROSS DATASETS
# =====================================================================

def verify_no_duplicates(pool, tests, output):
    frames = [
        pool.assign(dataset="pooled")
    ]

    frames += [
        df.assign(dataset=f"test_{patient}")
        for patient, df in tests.items()
    ]

    combined = pd.concat(
        frames,
        ignore_index=True,
    )

    repeated = combined[
        combined.duplicated(
            "pixel_sha256",
            keep=False,
        )
    ]

    if not repeated.empty:
        report = output / "duplicate_image_report.csv"

        repeated.to_csv(report, index=False)

        raise ValueError(
            "Identical image pixels were found within/across datasets.\n"
            f"Inspect:\n{report}\n"
            "No images were removed automatically. Resolve duplicate "
            "copies or investigate their provenance before training. "
            "Matching pixels alone do not establish EEG-window identity."
        )


# =====================================================================
# 8. CREATE CLASS-STRATIFIED TRAIN / VALIDATION SPLIT
# =====================================================================

def split_pool(pool):
    training_parts = []
    validation_parts = []

    for label in (0, 1):
        group = pool[
            pool["label"] == label
        ].sample(
            frac=1,
            random_state=SEED,
        )

        if len(group) < 2:
            raise ValueError(
                "Need at least two pooled examples per class."
            )

        validation_count = max(
            1,
            min(
                len(group) - 1,
                math.ceil(len(group) * VAL_FRACTION),
            ),
        )

        validation_parts.append(
            group.iloc[-validation_count:]
        )

        training_parts.append(
            group.iloc[:-validation_count]
        )

    training_df = pd.concat(
        training_parts
    ).sample(
        frac=1,
        random_state=SEED,
    ).reset_index(drop=True)

    validation_df = pd.concat(
        validation_parts
    ).sample(
        frac=1,
        random_state=SEED,
    ).reset_index(drop=True)

    return training_df, validation_df


# =====================================================================
# 9. DATASET
# =====================================================================

class Topomaps(Dataset):
    def __init__(self, df):
        self.rows = df.to_dict("records")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


# =====================================================================
# 10. CHAT MESSAGES
# =====================================================================

def messages(image):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": USER_PROMPT,
                },
                {
                    "type": "image",
                    "image": image,
                },
            ],
        },
    ]


# =====================================================================
# 11. COLLATOR WITH ASSISTANT-ONLY LOSS
# =====================================================================

class Collator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, rows):
        if len(rows) != 1:
            raise ValueError(
                "This collator requires physical batch size 1."
            )

        row = rows[0]

        with Image.open(row["image_path"]) as opened:
            image = opened.convert("RGB")

            chat = messages(image)

            prompt = self.processor.apply_chat_template(
                chat,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

            target_text = (
                "Seizure"
                if row["label"] == 1
                else "Non-seizure"
            )

            chat.append(
                {
                    "role": "assistant",
                    "content": [
                        {
                            "type": "text",
                            "text": target_text,
                        }
                    ],
                }
            )

            full = self.processor.apply_chat_template(
                chat,
                add_generation_prompt=False,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

        prompt_length = prompt["input_ids"].shape[1]

        if (
            full["input_ids"].shape[1] <= prompt_length
            or not torch.equal(
                full["input_ids"][:, :prompt_length],
                prompt["input_ids"],
            )
        ):
            raise RuntimeError(
                "Chat-template prefix mismatch. "
                "Assistant-only loss mask would be invalid."
            )

        full["labels"] = full["input_ids"].clone()
        full["labels"][:, :prompt_length] = -100

        if "attention_mask" in full:
            full["labels"][
                full["attention_mask"] == 0
            ] = -100

        return full


# =====================================================================
# 12. MOVE INPUTS TO GPU
# =====================================================================

def move(batch, dtype):
    return {
        key: (
            value.to(
                "cuda:0",
                dtype=(
                    dtype
                    if value.is_floating_point()
                    else value.dtype
                ),
            )
            if torch.is_tensor(value)
            else value
        )
        for key, value in batch.items()
    }


# =====================================================================
# 13. LOAD BASE MODEL
# =====================================================================

def load_base(dtype):
    model_class = getattr(
        transformers,
        "AutoModelForImageTextToText",
        None,
    )

    if model_class is None:
        model_class = getattr(
            transformers,
            "AutoModelForMultimodalLM",
            None,
        )

    if model_class is None:
        raise RuntimeError(
            "Your Transformers installation lacks a supported "
            "multimodal auto-model class."
        )

    model = model_class.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        token=True,
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )

    return model.to("cuda:0")


# =====================================================================
# 14. PARSE GENERATED LABEL
# =====================================================================

def prediction_label(text):
    normalized = re.sub(
        r"[\s_\-–]+",
        "",
        text.strip().lower().strip(". !\n\t"),
    )

    return {
        "seizure": 1,
        "nonseizure": 0,
    }.get(normalized)


# =====================================================================
# 15. GENERATION-BASED EVALUATION
# =====================================================================

@torch.no_grad()
def evaluate(model, processor, df, dtype, description):
    model.eval()

    raw_outputs = []
    predictions = []

    for row in tqdm(
        df.to_dict("records"),
        desc=description,
    ):
        with Image.open(row["image_path"]) as opened:
            inputs = processor.apply_chat_template(
                messages(opened.convert("RGB")),
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )

        prompt_length = inputs["input_ids"].shape[1]

        with torch.autocast("cuda", dtype=dtype):
            generated = model.generate(
                **move(inputs, dtype),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                use_cache=True,
            )

        text = processor.decode(
            generated[0, prompt_length:],
            skip_special_tokens=True,
        ).strip()

        raw_outputs.append(text)
        predictions.append(prediction_label(text))

    true_labels = df["label"].to_numpy(dtype=int)

    scored_predictions = np.array(
        [
            1 - int(truth) if prediction is None else prediction
            for truth, prediction in zip(true_labels, predictions)
        ]
    )

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        scored_predictions,
        labels=[0, 1],
    ).ravel()

    metrics = {
        "n_images": len(df),

        "accuracy": float(
            accuracy_score(
                true_labels,
                scored_predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                true_labels,
                scored_predictions,
            )
        ),

        "sensitivity": safe_divide(tp, tp + fn),
        "specificity": safe_divide(tn, tn + fp),

        "precision": float(
            precision_score(
                true_labels,
                scored_predictions,
                zero_division=0,
            )
        ),

        "f1": float(
            f1_score(
                true_labels,
                scored_predictions,
                zero_division=0,
            )
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),

        "invalid_predictions": sum(
            prediction is None
            for prediction in predictions
        ),

        "invalid_output_policy": (
            "count_as_wrong_for_metrics_only"
        ),
    }

    result = df.copy()

    result["raw_output"] = raw_outputs

    result["predicted_label"] = pd.array(
        predictions,
        dtype="Int64",
    )

    result["invalid_output"] = [
        prediction is None
        for prediction in predictions
    ]

    result[
        "metric_label_invalid_counted_wrong"
    ] = scored_predictions

    result["correct"] = [
        prediction is not None and prediction == truth
        for prediction, truth in zip(predictions, true_labels)
    ]

    return metrics, result


# =====================================================================
# 16. VALIDATION LOSS
# =====================================================================

@torch.no_grad()
def validation_loss(model, loader, dtype):
    model.eval()

    total_loss = 0.0

    for batch in tqdm(
        loader,
        desc="Validation loss",
    ):
        with torch.autocast("cuda", dtype=dtype):
            loss = model(
                **move(batch, dtype)
            ).loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Nonfinite validation loss."
            )

        total_loss += loss.item()

    return total_loss / len(loader)


# =====================================================================
# 17. MAIN PIPELINE
# =====================================================================

def main():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU required. Run in your existing CUDA environment."
        )

    seed_all()

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    print("GPU:", torch.cuda.get_device_name(0))
    print("Precision:", dtype)

    output = OUTPUT_BASE / datetime.now().strftime(
        "run_%Y%m%d_%H%M%S_%f"
    )

    output.mkdir(
        parents=True,
        exist_ok=False,
    )

    best_dir = output / "best_lora_adapter"

    print("\nOutput folder:")
    print(output)

    pool = load_metadata(
        TRAIN_CSV,
        "pooled",
    )

    tests = {
        patient: load_metadata(csv_path, patient)
        for patient, csv_path in TEST_CSVS.items()
    }

    verify_no_duplicates(
        pool,
        tests,
        output,
    )

    train_df, val_df = split_pool(pool)

    train_df.to_csv(
        output / "actual_train.csv",
        index=False,
    )

    val_df.to_csv(
        output / "validation.csv",
        index=False,
    )

    for name, current_df in [
        ("TRAIN", train_df),
        ("VALIDATION", val_df),
    ]:
        print(
            f"\n{name}: {len(current_df)} images"
            f"\nClass counts: {current_df['label'].value_counts().to_dict()}"
        )

    config = {
        "model_id": MODEL_ID,
        "train_csv": TRAIN_CSV,
        "test_csvs": TEST_CSVS,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "validation_fraction": VAL_FRACTION,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "max_grad_norm": MAX_GRAD_NORM,
        "warmup_ratio": WARMUP_RATIO,
        "max_new_tokens": MAX_NEW_TOKENS,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": USER_PROMPT,
        "precision": str(dtype),
        "gpu": torch.cuda.get_device_name(0),
        "python": sys.version,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "checkpoint_rule": (
            "Highest validation BA; lower validation loss breaks ties."
        ),
        "stopping_rule": (
            "7 consecutive epochs without strict validation BA improvement."
        ),
        "split_rule": (
            "Class-stratified random image split, approximately 80/20."
        ),
        "representation": "A1",
    }

    save_json(
        output / "experiment_config.json",
        config,
    )

    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        token=True,
    )

    base = load_base(dtype)

    target_leaves = {
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    }

    candidates = [
        name
        for name, module in base.named_modules()
        if (
            isinstance(module, torch.nn.Linear)
            and name.split(".")[-1] in target_leaves
            and "vision" not in name.lower()
        )
    ]

    language_targets = [
        name
        for name in candidates
        if "language_model" in name.lower()
    ]

    targets = language_targets or candidates

    if not targets:
        raise RuntimeError(
            "No language attention projections found."
        )

    (output / "target_modules.txt").write_text(
        "\n".join(targets),
        encoding="utf-8",
    )

    model = get_peft_model(
        base,
        LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=targets,
            bias="none",
            task_type="CAUSAL_LM",
        ),
    )

    del base

    model.config.use_cache = False

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False,
        }
    )

    model.print_trainable_parameters()

    params = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if any(
        "vision" in name.lower() and parameter.requires_grad
        for name, parameter in model.named_parameters()
    ):
        raise RuntimeError(
            "Unexpected trainable vision parameters."
        )

    config["trainable_parameters"] = sum(
        parameter.numel()
        for parameter in params
    )

    config["total_parameters_including_adapters"] = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    config["number_of_target_modules"] = len(targets)

    config["base_commit"] = getattr(
        model.config,
        "_commit_hash",
        None,
    )

    save_json(
        output / "experiment_config.json",
        config,
    )

    generator = torch.Generator().manual_seed(SEED)

    collator = Collator(processor)

    train_loader = DataLoader(
        Topomaps(train_df),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
    )

    val_loader = DataLoader(
        Topomaps(val_df),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=collator,
    )

    optimizer = torch.optim.AdamW(
        params,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = (
        math.ceil(len(train_loader) / GRAD_ACCUM)
        * MAX_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=math.ceil(
            total_steps * WARMUP_RATIO
        ),
        num_training_steps=total_steps,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(dtype == torch.float16),
    )

    best_ba = -float("inf")
    best_loss = float("inf")
    best_epoch = None

    stale_epochs = 0
    history = []

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    training_start = time.perf_counter()

    for epoch in range(1, MAX_EPOCHS + 1):
        epoch_start = time.perf_counter()

        model.train()
        optimizer.zero_grad(set_to_none=True)

        total_loss = 0.0

        for step, batch in enumerate(
            tqdm(
                train_loader,
                desc=f"Epoch {epoch}/{MAX_EPOCHS}",
            )
        ):
            group_start = (
                step // GRAD_ACCUM
            ) * GRAD_ACCUM

            group_size = min(
                GRAD_ACCUM,
                len(train_loader) - group_start,
            )

            with torch.autocast("cuda", dtype=dtype):
                loss = model(
                    **move(batch, dtype)
                ).loss

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Nonfinite training loss at "
                    f"epoch {epoch}, step {step}."
                )

            total_loss += loss.item()

            scaler.scale(
                loss / group_size
            ).backward()

            update_now = (
                (step + 1) % GRAD_ACCUM == 0
                or step + 1 == len(train_loader)
            )

            if update_now:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    params,
                    MAX_GRAD_NORM,
                )

                previous_scale = scaler.get_scale()

                scaler.step(optimizer)
                scaler.update()

                if scaler.get_scale() >= previous_scale:
                    scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        current_val_loss = validation_loss(
            model,
            val_loader,
            dtype,
        )

        val_metrics, val_predictions = evaluate(
            model,
            processor,
            val_df,
            dtype,
            "Validation generation",
        )

        current_ba = val_metrics["balanced_accuracy"]

        improved_ba = current_ba > best_ba + 1e-12

        tied_ba = math.isclose(
            current_ba,
            best_ba,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

        improved_checkpoint = (
            improved_ba
            or (
                tied_ba
                and current_val_loss < best_loss
            )
        )

        if improved_ba:
            stale_epochs = 0
        else:
            stale_epochs += 1

        if improved_checkpoint:
            best_ba = current_ba
            best_loss = current_val_loss
            best_epoch = epoch

            model.save_pretrained(
                best_dir,
                safe_serialization=True,
            )

            processor.save_pretrained(best_dir)

            val_predictions.to_csv(
                output / "best_validation_predictions.csv",
                index=False,
            )

            save_json(
                output / "best_checkpoint.json",
                {
                    "best_epoch": epoch,
                    "validation_balanced_accuracy": current_ba,
                    "validation_loss": current_val_loss,
                    "adapter_path": best_dir,
                },
            )

        torch.cuda.synchronize()

        epoch_record = {
            "epoch": epoch,
            "training_loss": total_loss / len(train_loader),
            "validation_loss": current_val_loss,

            **{
                f"validation_{key}": value
                for key, value in val_metrics.items()
            },

            "saved_best": improved_checkpoint,
            "epochs_without_ba_improvement": stale_epochs,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.perf_counter() - epoch_start,
        }

        history.append(epoch_record)

        pd.DataFrame(history).to_csv(
            output / "training_history.csv",
            index=False,
        )

        print(
            f"\nEpoch {epoch}: "
            f"train loss={epoch_record['training_loss']:.5f}, "
            f"val loss={current_val_loss:.5f}, "
            f"val BA={current_ba:.4%}"
        )

        print(
            f"Best epoch={best_epoch}; "
            f"patience={stale_epochs}/{PATIENCE}"
        )

        if stale_epochs >= PATIENCE:
            print(
                "\nEarly stopping: validation BA did not "
                f"improve for {PATIENCE} consecutive epochs."
            )
            break

    torch.cuda.synchronize()

    save_json(
        output / "training_summary.json",
        {
            "best_epoch": best_epoch,
            "best_validation_ba": best_ba,
            "epochs_completed": len(history),

            "training_and_validation_seconds": (
                time.perf_counter() - training_start
            ),

            "peak_allocated_vram_gib": (
                torch.cuda.max_memory_allocated() / 1024**3
            ),

            "peak_reserved_vram_gib": (
                torch.cuda.max_memory_reserved() / 1024**3
            ),
        },
    )

    del model
    del optimizer
    del scheduler
    del scaler
    del params

    if "loss" in locals():
        del loss

    if "batch" in locals():
        del batch

    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"\nReloading best checkpoint from epoch {best_epoch}."
    )

    model = PeftModel.from_pretrained(
        load_base(dtype),
        best_dir,
        is_trainable=False,
    )

    model.eval()
    model.config.use_cache = True

    summary = []

    for patient, test_df in tests.items():
        patient_dir = output / f"Patient_{patient}"

        patient_dir.mkdir(exist_ok=True)

        metrics, predictions = evaluate(
            model,
            processor,
            test_df,
            dtype,
            f"Test patient {patient}",
        )

        metrics.update(
            {
                "patient": patient,
                "best_epoch": best_epoch,
                "test_csv": str(TEST_CSVS[patient]),
            }
        )

        predictions.to_csv(
            patient_dir / "predictions.csv",
            index=False,
        )

        save_json(
            patient_dir / "metrics.json",
            metrics,
        )

        matrix_df = pd.DataFrame(
            [
                [metrics["TN"], metrics["FP"]],
                [metrics["FN"], metrics["TP"]],
            ],
            index=[
                "true_non_seizure",
                "true_seizure",
            ],
            columns=[
                "scored_non_seizure",
                "scored_seizure",
            ],
        )

        matrix_df.to_csv(
            patient_dir / "confusion_matrix.csv"
        )

        summary.append(metrics)

        pd.DataFrame(summary).to_csv(
            output / "patientwise_test_summary.csv",
            index=False,
        )

        print(
            f"\nPatient {patient}: "
            f"Accuracy={metrics['accuracy']:.2%}, "
            f"BA={metrics['balanced_accuracy']:.2%}, "
            f"Sensitivity={metrics['sensitivity']:.2%}, "
            f"Specificity={metrics['specificity']:.2%}, "
            f"F1={metrics['f1']:.4f}"
        )

    print(
        "\nCompleted. The same best A1 LoRA adapter was evaluated "
        "on all four patient test sets."
    )

    print("\nResults saved in:")
    print(output)


# =====================================================================
# 18. RUN
# =====================================================================

if __name__ == "__main__":
    main()


W1005 15:49:39.954000 29596 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


GPU: NVIDIA GeForce RTX 5080
Precision: torch.bfloat16

Output folder:
C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A1\MedGemma4B_LoRA_Pooled_A1\run_20261005_154941_574007


Validate images: pooled: 100%|██████████| 2875/2875 [00:48<00:00, 59.84it/s]



pooled: 2875 images
Class counts: {0: 2300, 1: 575}


Validate images: 08: 100%|██████████| 1000/1000 [00:14<00:00, 66.84it/s]



08: 1000 images
Class counts: {0: 800, 1: 200}
11: attached labels from patient11_A1_blinded_labels.csv using key 'blind_id'.


... (46 outputs left out)

Test patient 18: 100%|██████████| 375/375 [03:30<00:00,  1.78it/s]



Patient 18: Accuracy=94.40%, BA=92.50%, Sensitivity=89.33%, Specificity=95.67%, F1=0.8645


Test patient 24: 100%|██████████| 500/500 [04:45<00:00,  1.75it/s]



Patient 24: Accuracy=90.80%, BA=82.62%, Sensitivity=69.00%, Specificity=96.25%, F1=0.7500

Completed. The same best A1 LoRA adapter was evaluated on all four patient test sets.

Results saved in:
C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A1\MedGemma4B_LoRA_Pooled_A1\run_20261005_154941_574007


In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# RUN FOLDER
# ============================================================

RUN_DIR = Path(
    r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\A1\MedGemma4B_LoRA_Pooled_A1\run_20261005_154941_574007"
)

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run folder not found:\n{RUN_DIR}")


# ============================================================
# HELPERS
# ============================================================

def compute_confusion_from_predictions(predictions_csv):
    df = pd.read_csv(predictions_csv)

    if "label" not in df.columns:
        raise ValueError(f"No label column found in:\n{predictions_csv}")

    true = df["label"].astype(int).to_numpy()

    # This column was created by your training code.
    # It already counts invalid outputs as wrong.
    if "metric_label_invalid_counted_wrong" in df.columns:
        pred = df["metric_label_invalid_counted_wrong"].astype(int).to_numpy()

    elif "predicted_label" in df.columns:
        pred_series = pd.to_numeric(df["predicted_label"], errors="coerce")

        # If prediction is invalid/missing, count it as wrong.
        pred = []
        for y_true, y_pred in zip(true, pred_series):
            if pd.isna(y_pred):
                pred.append(1 - int(y_true))
            else:
                pred.append(int(y_pred))

        pred = np.asarray(pred, dtype=int)

    else:
        raise ValueError(
            f"No prediction column found in:\n{predictions_csv}"
        )

    tn = int(((true == 0) & (pred == 0)).sum())
    fp = int(((true == 0) & (pred == 1)).sum())
    fn = int(((true == 1) & (pred == 0)).sum())
    tp = int(((true == 1) & (pred == 1)).sum())

    return tn, fp, fn, tp


def safe_div(a, b):
    return float(a / b) if b != 0 else float("nan")


def metrics_from_matrix(tn, fp, fn, tp):
    accuracy = safe_div(tp + tn, tp + tn + fp + fn)
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    balanced_accuracy = (sensitivity + specificity) / 2

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1": f1,
    }


def make_matrix_df(tn, fp, fn, tp):
    return pd.DataFrame(
        [
            [tn, fp],
            [fn, tp],
        ],
        index=[
            "true_non_seizure",
            "true_seizure",
        ],
        columns=[
            "pred_non_seizure",
            "pred_seizure",
        ],
    )


# ============================================================
# READ / RECONSTRUCT CONFUSION MATRICES
# ============================================================

patient_dirs = sorted(RUN_DIR.glob("Patient_*"))

if not patient_dirs:
    raise FileNotFoundError(
        f"No Patient_* folders found inside:\n{RUN_DIR}"
    )

summary_records = []
long_records = []

for patient_dir in patient_dirs:
    patient = patient_dir.name.replace("Patient_", "")

    predictions_csv = patient_dir / "predictions.csv"
    saved_matrix_csv = patient_dir / "confusion_matrix.csv"

    if predictions_csv.exists():
        tn, fp, fn, tp = compute_confusion_from_predictions(predictions_csv)

    elif saved_matrix_csv.exists():
        saved_matrix = pd.read_csv(saved_matrix_csv, index_col=0)

        tn = int(saved_matrix.iloc[0, 0])
        fp = int(saved_matrix.iloc[0, 1])
        fn = int(saved_matrix.iloc[1, 0])
        tp = int(saved_matrix.iloc[1, 1])

    else:
        raise FileNotFoundError(
            f"No predictions.csv or confusion_matrix.csv found in:\n{patient_dir}"
        )

    matrix_df = make_matrix_df(tn, fp, fn, tp)

    # Save a clean version
    clean_matrix_csv = patient_dir / "confusion_matrix_clean.csv"
    matrix_df.to_csv(clean_matrix_csv)

    metric_values = metrics_from_matrix(tn, fp, fn, tp)

    summary_records.append({
        "patient": patient,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": metric_values["accuracy"],
        "balanced_accuracy": metric_values["balanced_accuracy"],
        "sensitivity": metric_values["sensitivity"],
        "specificity": metric_values["specificity"],
        "precision": metric_values["precision"],
        "f1": metric_values["f1"],
        "matrix_csv": str(clean_matrix_csv),
    })

    long_records.extend([
        {
            "patient": patient,
            "true_class": "non_seizure",
            "predicted_class": "non_seizure",
            "count": tn,
        },
        {
            "patient": patient,
            "true_class": "non_seizure",
            "predicted_class": "seizure",
            "count": fp,
        },
        {
            "patient": patient,
            "true_class": "seizure",
            "predicted_class": "non_seizure",
            "count": fn,
        },
        {
            "patient": patient,
            "true_class": "seizure",
            "predicted_class": "seizure",
            "count": tp,
        },
    ])

    print("\n" + "=" * 60)
    print(f"Patient {patient}")
    print("=" * 60)
    print(matrix_df)

    print(
        f"\nAccuracy: {metric_values['accuracy']:.2%} | "
        f"BA: {metric_values['balanced_accuracy']:.2%} | "
        f"Sensitivity: {metric_values['sensitivity']:.2%} | "
        f"Specificity: {metric_values['specificity']:.2%} | "
        f"F1: {metric_values['f1']:.4f}"
    )


# ============================================================
# SAVE COMBINED SUMMARY
# ============================================================

summary_df = pd.DataFrame(summary_records)
long_df = pd.DataFrame(long_records)

summary_csv = RUN_DIR / "all_patient_confusion_matrix_summary.csv"
long_csv = RUN_DIR / "all_patient_confusion_matrix_long_format.csv"

summary_df.to_csv(summary_csv, index=False)
long_df.to_csv(long_csv, index=False)

print("\n\nDONE.")
print("Summary saved:")
print(summary_csv)

print("\nLong-format confusion matrix saved:")
print(long_csv)

print("\nSummary table:")
display(summary_df)


Patient 08
                  pred_non_seizure  pred_seizure
true_non_seizure               742            58
true_seizure                    23           177

Accuracy: 91.90% | BA: 90.62% | Sensitivity: 88.50% | Specificity: 92.75% | F1: 0.8138

Patient 11
                  pred_non_seizure  pred_seizure
true_non_seizure               783            17
true_seizure                    15           185

Accuracy: 96.80% | BA: 95.19% | Sensitivity: 92.50% | Specificity: 97.88% | F1: 0.9204

Patient 18
                  pred_non_seizure  pred_seizure
true_non_seizure               287            13
true_seizure                     8            67

Accuracy: 94.40% | BA: 92.50% | Sensitivity: 89.33% | Specificity: 95.67% | F1: 0.8645

Patient 24
                  pred_non_seizure  pred_seizure
true_non_seizure               385            15
true_seizure                    31            69

Accuracy: 90.80% | BA: 82.62% | Sensitivity: 69.00% | Specificity: 96.25% | F1: 0.7500


DONE.
Summ

,patient,TN,FP,FN,TP,accuracy,balanced_accuracy,sensitivity,specificity,precision,f1,matrix_csv
0,08,742,58,23,177,0.919,0.906250,0.885000,0.927500,0.753191,0.813793,C:\Users\ga_sd360\OneDrive - The University of...
1,11,783,17,15,185,0.968,0.951875,0.925000,0.978750,0.915842,0.920398,C:\Users\ga_sd360\OneDrive - The University of...
2,18,287,13,8,67,0.944,0.925000,0.893333,0.956667,0.837500,0.864516,C:\Users\ga_sd360\OneDrive - The University of...
3,24,385,15,31,69,0.908,0.826250,0.690000,0.962500,0.821429,0.750000,C:\Users\ga_sd360\OneDrive - The University of...
